## 1.项目初始化

### 1.导入库文件

In [1]:
import json  # 写 metrics.json、特征清单
import sqlite3  # 读 SQLite 数据库（Python 自带，不用装）

import joblib  # 保存整条 Pipeline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer  # 数值列、类别列分开处理
from sklearn.ensemble import RandomForestClassifier  # 随机森林
from sklearn.impute import SimpleImputer  # 填补缺失值
from sklearn.linear_model import LogisticRegression  # 逻辑回归
from sklearn.metrics import (  # 三个常用指标
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)  # 混淆矩阵、分类报告
from sklearn.model_selection import train_test_split  # 划分数据集用的
from sklearn.pipeline import Pipeline  # 搭建流水线
from sklearn.preprocessing import (
    OneHotEncoder,  # 对数据做独热编码
    StandardScaler,  # 对数据做标准化
)

pd.set_option("display.max_columns", 40)  # 列多，全部显示
pd.set_option("display.width", 120)


### 2.常量设置

In [17]:
SEED = 42
DATA_PATH = "../data/"
DB_PATH = DATA_PATH + "incident.db"
TARGET = 'label'


### 3.数据信息查看

In [36]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print(tables)

    for name in tables['name']:
        cols = pd.read_sql(f"PRAGMA table_info({name})", conn)
        n = pd.read_sql(f"select count(1) as n from {name}", conn)['n'][0]
        print(f"{name:<12} {n:>6} rows, {len(cols):>3} cols, {cols['name'].tolist()}")


          name
0    residents
1       visits
2  lab_results
residents     36000 rows,   7 cols, ['resident_id', 'gender', 'birth_date', 'education', 'income_level', 'district', 'enroll_date']
visits       101297 rows,  23 cols, ['visit_id', 'resident_id', 'visit_date', 'height_cm', 'weight_kg', 'waist_cm', 'sbp', 'dbp', 'smoking_status', 'drinking_freq', 'exercise_min_week', 'sleep_hours', 'veg_fruit_daily', 'family_history_dm', 'hypertension_dx', 'hyperlipidemia_dx', 'gdm_history', 'polyuria', 'polydipsia', 'weight_loss_unexplained', 'known_diabetes', 'created_at', 'chronic_mgmt_tag']
lab_results   79358 rows,   6 cols, ['lab_id', 'resident_id', 'sample_date', 'report_date', 'fpg_mmol_l', 'hba1c_pct']


### 4.数据读取

In [101]:
with sqlite3.connect(DB_PATH) as conn:
    residents = pd.read_sql("select * from residents", conn, parse_dates=["birth_date", "enroll_date"])
    visits = pd.read_sql("select * from visits", conn, parse_dates=["visit_date", 'created_at'])
    lab_results = pd.read_sql("select * from lab_results", conn, parse_dates=["sample_date", "report_date"])
    print(
        residents.shape,
        visits.shape,
        lab_results.shape,
    )


(36000, 7) (101297, 23) (79358, 6)


In [ ]:
visits.head(3).T

In [ ]:
visits.describe(include='number').T[['count', 'mean', 'min', '50%', 'max']].round(1)


In [105]:
print(visits.isna().mean().round(3).sort_values(ascending=False).head(4))
print('完全重复的行：', visits.duplicated().sum())
# 根据性别统计行数
print(residents['gender'].value_counts().to_dict())
# 查看吸烟状态
print(visits['smoking_status'].value_counts().to_dict())
# 查看饮酒状态
print(visits['drinking_freq'].value_counts().to_dict())

gdm_history                0.485
waist_cm                   0.151
sleep_hours                0.080
weight_loss_unexplained    0.030
dtype: float64
完全重复的行： 0
{'F': 18622, 'M': 17378}
{'never': 69342, 'current': 22966, 'former': 8989}
{'never': 51058, 'occasional': 35334, 'frequent': 14905}


## 2.数据清洗

In [ ]:
# 删除完全重复的行
visits = visits.drop_duplicates()
# 统一性别类别标签
residents['gender'] = residents['gender'].replace(
    {'male': 'M', 'female': 'F', '男': 'M', '女': 'F'}
)

visits['smoking_status'] = visits['smoking_status'].replace(
    {
        '从不': 'never',
        '吸烟': 'current',
        '已戒': 'former',
        'Former': 'former',
        'Never': 'never',
        'Current': 'current',
        'never ': 'never',
    }
)

visits['drinking_freq'] = visits['drinking_freq'].replace(
    {
        '不喝': 'never',
        '偶尔': 'occasional',
        '经常': 'frequent',
        'Never': 'never',
        'Occasional': 'occasional',
        'Frequent': 'frequent',
    }
)

# 将身高单位从米改为厘米
visits.loc[visits['height_cm'] < 3, 'height_cm'] *= 100
visits[visits['height_cm'] < 3]  # 检查是否还有小于3米的身高

# 将体重单位从斤改为公斤
visits.loc[visits['weight_kg'] > 150, 'weight_kg'] /= 2
visits[visits['weight_kg'] > 150]  # 检查是否还有大于150公斤的体重


# 将收缩压为0的记录设为缺失值
visits.loc[visits['sbp'] == 0, 'sbp'] = np.nan
visits[visits['sbp'] == 0]  # 检查是否还有收缩压为0的记录


# 将运动时间超过3000分钟的记录设为缺失值
visits.loc[visits['exercise_min_week'] > 3000, 'exercise_min_week'] = np.nan
visits[visits['exercise_min_week'] > 3000]  # 检查是否还有运动时间超过3000分钟的记录



,visit_id,resident_id,visit_date,height_cm,weight_kg,waist_cm,sbp,dbp,smoking_status,drinking_freq,exercise_min_week,sleep_hours,veg_fruit_daily,family_history_dm,hypertension_dx,hyperlipidemia_dx,gdm_history,polyuria,polydipsia,weight_loss_unexplained,known_diabetes,created_at,chronic_mgmt_tag
